**One-time setup notebook.** Run all cells to generate the `data/` folder used by every course notebook.

- Seeded with `random.seed(42)` — re-running always produces the same dataset
- Writes 8 tables across 5 formats: CSV, JSON, Parquet, ORC, Delta
- `data/` is in `.gitignore` and is never committed to the repo

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("GenerateBankData")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession ready.")

26/04/08 10:54:21 WARN Utils: Your hostname, maddipotis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.181 instead (on interface en0)
26/04/08 10:54:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/maddipotiganesh/.ivy2/cache
The jars for the packages stored in: /Users/maddipotiganesh/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-188df9c6-bca2-4833-8678-22f4f02aacbb;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...


	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (1543ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.1/delta-storage-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.1!delta-storage.jar (79ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...


	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (146ms)
:: resolution report :: resolve 1500ms :: artifacts dl 1773ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   3   |   3   |   0   ||   3   |   3   |
	---------------------------------------------------------------------

:: problems summary ::
:::: ERRORS
	unknown resolver Maven Central repository

	unknown resolver Maven Central repository


:: USE VERBOSE OR DEBUG MESSAGE LEVEL FOR MORE DETAILS
:: retrieving :: org.apache.spark#spark-submit-parent-188df9c6-bca

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SparkSession ready.


In [2]:
import os, random, decimal
from datetime import date, datetime, timedelta
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, BooleanType,
    DecimalType, DateType, TimestampType, DoubleType,
)

random.seed(42)

# ── helpers ────────────────────────────────────────────────────────────────
def uid(prefix, n):  return f"{prefix}{str(n).zfill(4)}"
def d(s):            return date.fromisoformat(s)
def money(lo, hi):   return decimal.Decimal(str(round(random.uniform(lo, hi), 2)))
def rand_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))
def rand_ts(s, e):
    a, b = datetime.fromisoformat(s), datetime.fromisoformat(e)
    return a + timedelta(seconds=random.randint(0, int((b - a).total_seconds())))
def maybe(val, p=0.10):
    return None if random.random() < p else val

# ── reference data ─────────────────────────────────────────────────────────
NAMES = [
    "Aarav Sharma","Priya Nair","Rohan Gupta","Anjali Mehta","Vikram Reddy",
    "Sunita Patel","Arjun Iyer","Deepika Joshi","Kabir Singh","Neha Verma",
    "Amit Kumar","Pooja Desai","Sanjay Rao","Kavitha Pillai","Rahul Bose",
    "Meena Krishnan","Suresh Choudhary","Ritu Saxena","Manish Agarwal","Divya Nanda",
    "Kiran Malhotra","Sneha Tiwari","Ajay Ghosh","Lakshmi Subramanian","Tarun Mittal",
    "Bharti Pandey","Nikhil Jain","Shweta Kapoor","Gaurav Bansal","Pallavi Dubey",
    "Vijay Menon","Radha Naidu","Aryan Sethi","Smita Bhatt","Rakesh Chauhan",
    "Geeta Srivastava","Dev Anand","Falguni Shah","Pratik More","Rupal Doshi",
    "Chirag Parekh","Nidhi Goyal","Sameer Hegde","Tara Kulkarni","Vinay Garg",
    "Uma Rawat","Hitesh Thakur","Jasmine Oberoi","Lalit Yadav","Mona Sood",
]
CITIES_STATES = [
    ("Mumbai","Maharashtra"),("Delhi","Delhi"),("Bengaluru","Karnataka"),
    ("Hyderabad","Telangana"),("Chennai","Tamil Nadu"),("Pune","Maharashtra"),
    ("Ahmedabad","Gujarat"),("Kolkata","West Bengal"),("Jaipur","Rajasthan"),
    ("Lucknow","Uttar Pradesh"),
]
MERCHANTS  = ["Amazon","Swiggy","Zomato","BigBasket","Flipkart",
              "Ola","Uber","BookMyShow","MakeMyTrip","IRCTC"]
CATEGORIES = ["Shopping","Food","Groceries","Transport","Entertainment","Travel"]
IFSC_PFX   = ["HDFC","ICIC","SBIN","AXIS","KOTK"]
DESCS      = ["Salary credit","UPI transfer","EMI debit","ATM withdrawal",
              "Cheque deposit","FD interest","Bill payment","NEFT transfer"]

def ifsc(p): return f"{p}0{random.randint(100000,999999)}"

N = 50
customer_ids = [uid("CUST", i+1) for i in range(N)]
print(f"Reference data ready — {N} customers.")

Reference data ready — 50 customers.


In [3]:
# ── schemas (same as 00-bank-domain.ipynb) ──────────────────────────────────
customers_schema = StructType([
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("full_name",     StringType(),       nullable=False),
    StructField("email",         StringType(),       nullable=True),
    StructField("phone",         StringType(),       nullable=True),
    StructField("city",          StringType(),       nullable=True),
    StructField("state",         StringType(),       nullable=True),
    StructField("date_of_birth", DateType(),         nullable=True),
    StructField("kyc_verified",  BooleanType(),      nullable=False),
    StructField("created_at",    TimestampType(),    nullable=False),
])
card_accounts_schema = StructType([
    StructField("card_id",       StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("card_type",     StringType(),       nullable=False),
    StructField("network",       StringType(),       nullable=True),
    StructField("credit_limit",  DecimalType(18,2),  nullable=True),
    StructField("issued_on",     DateType(),         nullable=False),
    StructField("expiry_date",   DateType(),         nullable=False),
    StructField("status",        StringType(),       nullable=False),
])
card_transactions_schema = StructType([
    StructField("txn_id",        StringType(),       nullable=False),
    StructField("card_id",       StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("amount",        DecimalType(18,2),  nullable=False),
    StructField("merchant",      StringType(),       nullable=True),
    StructField("category",      StringType(),       nullable=True),
    StructField("txn_ts",        TimestampType(),    nullable=False),
    StructField("status",        StringType(),       nullable=False),
    StructField("is_fraud",      BooleanType(),      nullable=False),
])
loan_accounts_schema = StructType([
    StructField("loan_id",       StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("loan_type",     StringType(),       nullable=False),
    StructField("principal",     DecimalType(18,2),  nullable=False),
    StructField("interest_rate", DoubleType(),       nullable=False),
    StructField("tenure_months", IntegerType(),      nullable=False),
    StructField("disbursed_on",  DateType(),         nullable=False),
    StructField("status",        StringType(),       nullable=False),
])
loan_repayments_schema = StructType([
    StructField("repayment_id",  StringType(),       nullable=False),
    StructField("loan_id",       StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("due_date",      DateType(),         nullable=False),
    StructField("paid_date",     DateType(),         nullable=True),
    StructField("emi_amount",    DecimalType(18,2),  nullable=False),
    StructField("paid_amount",   DecimalType(18,2),  nullable=True),
    StructField("status",        StringType(),       nullable=False),
])
bank_accounts_schema = StructType([
    StructField("account_id",    StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("account_type",  StringType(),       nullable=False),
    StructField("balance",       DecimalType(18,2),  nullable=False),
    StructField("branch_code",   StringType(),       nullable=True),
    StructField("ifsc",          StringType(),       nullable=True),
    StructField("opened_on",     DateType(),         nullable=False),
    StructField("is_active",     BooleanType(),      nullable=False),
])
account_transactions_schema = StructType([
    StructField("txn_id",        StringType(),       nullable=False),
    StructField("account_id",    StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("txn_type",      StringType(),       nullable=False),
    StructField("amount",        DecimalType(18,2),  nullable=False),
    StructField("balance_after", DecimalType(18,2),  nullable=False),
    StructField("description",   StringType(),       nullable=True),
    StructField("txn_ts",        TimestampType(),    nullable=False),
])
payments_schema = StructType([
    StructField("payment_id",    StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("payment_mode",  StringType(),       nullable=False),
    StructField("amount",        DecimalType(18,2),  nullable=False),
    StructField("sender_vpa",    StringType(),       nullable=True),
    StructField("receiver_vpa",  StringType(),       nullable=True),
    StructField("status",        StringType(),       nullable=False),
    StructField("is_fraud",      BooleanType(),      nullable=False),
    StructField("initiated_at",  TimestampType(),    nullable=False),
    StructField("settled_at",    TimestampType(),    nullable=True),
])
print("Schemas loaded.")

Schemas loaded.


In [4]:
# ── customers ──────────────────────────────────────────────────────────────
rows = []
for i, cid in enumerate(customer_ids):
    city, state = random.choice(CITIES_STATES)
    first = NAMES[i].split()[0].lower()
    rows.append((
        cid, NAMES[i],
        maybe(f"{first}.{random.randint(10,99)}@email.in"),
        maybe(f"9{random.randint(100000000,999999999)}"),
        city, state,
        rand_date(d("1960-01-01"), d("2000-12-31")),
        random.choice([True, True, True, False]),
        rand_ts("2020-01-01 00:00:00", "2024-06-01 00:00:00"),
    ))
df_customers = spark.createDataFrame(rows, schema=customers_schema)
print(f"customers         : {df_customers.count():>4} rows")

customers         :   50 rows


In [5]:
# ── card_accounts ──────────────────────────────────────────────────────────
rows, card_ids, card_to_cust = [], [], {}
c = 1
for cid in customer_ids:
    for _ in range(random.choices([1, 2], weights=[0.6, 0.4])[0]):
        cid2 = uid("CARD", c); card_ids.append(cid2); card_to_cust[cid2] = cid
        issued = rand_date(d("2019-01-01"), d("2023-12-31"))
        ct     = random.choice(["CREDIT", "DEBIT"])
        rows.append((
            cid2, cid, ct,
            maybe(random.choice(["VISA","MASTERCARD","RUPAY"])),
            decimal.Decimal(str(random.choice([50000,100000,150000,200000,300000]))) if ct=="CREDIT" else None,
            issued, date(issued.year+3, issued.month, 1),
            random.choice(["ACTIVE","ACTIVE","ACTIVE","BLOCKED","EXPIRED"]),
        ))
        c += 1
df_card_accounts = spark.createDataFrame(rows, schema=card_accounts_schema)
print(f"card_accounts     : {df_card_accounts.count():>4} rows")

card_accounts     :   67 rows


In [6]:
# ── card_transactions ──────────────────────────────────────────────────────
rows, c = [], 1
for card_id in card_ids:
    for _ in range(random.randint(2, 5)):
        rows.append((
            uid("CTXN", c), card_id, card_to_cust[card_id],
            money(50, 25000),
            maybe(random.choice(MERCHANTS)),
            maybe(random.choice(CATEGORIES)),
            rand_ts("2023-01-01 00:00:00", "2024-12-31 23:59:59"),
            random.choice(["SUCCESS","SUCCESS","SUCCESS","DECLINED","REVERSED"]),
            random.random() < 0.05,
        ))
        c += 1
df_card_txns = spark.createDataFrame(rows, schema=card_transactions_schema)
print(f"card_transactions : {df_card_txns.count():>4} rows")

card_transactions :  234 rows


In [7]:
# ── loan_accounts ──────────────────────────────────────────────────────────
rows, loan_ids, loan_to_cust = [], [], {}
c = 1
for cid in random.sample(customer_ids, 35):
    for _ in range(random.choices([1, 2], weights=[0.8, 0.2])[0]):
        lid = uid("LOAN", c); loan_ids.append(lid); loan_to_cust[lid] = cid
        lt  = random.choice(["PERSONAL","HOME","AUTO"])
        rows.append((
            lid, cid, lt,
            money(50000, 5000000) if lt=="HOME" else money(10000, 500000),
            round(random.uniform(7.5, 18.0), 2),
            random.choice([12,24,36,48,60,120,180,240]),
            rand_date(d("2019-01-01"), d("2024-01-01")),
            random.choice(["ACTIVE","ACTIVE","CLOSED","NPA"]),
        ))
        c += 1
df_loan_accounts = spark.createDataFrame(rows, schema=loan_accounts_schema)
print(f"loan_accounts     : {df_loan_accounts.count():>4} rows")

loan_accounts     :   41 rows


In [8]:
# ── loan_repayments ────────────────────────────────────────────────────────
rows, c = [], 1
for lid in loan_ids:
    cid = loan_to_cust[lid]
    for month in range(1, 4):
        due = d("2024-01-01") + timedelta(days=30*month)
        emi = money(3000, 25000)
        if random.random() < 0.08:
            rows.append((uid("REP",c), lid, cid, due, None, emi, None, "MISSED"))
        elif random.random() < 0.05:
            rows.append((uid("REP",c), lid, cid, due,
                         due+timedelta(days=random.randint(1,5)), emi,
                         money(float(emi)*0.5, float(emi)*0.9), "PARTIAL"))
        else:
            rows.append((uid("REP",c), lid, cid, due,
                         due+timedelta(days=random.randint(-2,2)), emi, emi, "PAID"))
        c += 1
df_repayments = spark.createDataFrame(rows, schema=loan_repayments_schema)
print(f"loan_repayments   : {df_repayments.count():>4} rows")

loan_repayments   :  123 rows


In [9]:
# ── bank_accounts ──────────────────────────────────────────────────────────
rows, account_ids, acct_to_cust = [], [], {}
for i, cid in enumerate(customer_ids):
    aid = uid("ACCT", i+1); account_ids.append(aid); acct_to_cust[aid] = cid
    rows.append((
        aid, cid,
        random.choice(["SAVINGS","SAVINGS","CURRENT"]),
        money(1000, 500000),
        maybe(f"BR{random.randint(100,999)}"),
        maybe(ifsc(random.choice(IFSC_PFX))),
        rand_date(d("2018-01-01"), d("2023-12-31")),
        random.choice([True,True,True,True,False]),
    ))
df_bank_accounts = spark.createDataFrame(rows, schema=bank_accounts_schema)
print(f"bank_accounts     : {df_bank_accounts.count():>4} rows")

bank_accounts     :   50 rows


In [10]:
# ── account_transactions ───────────────────────────────────────────────────
rows, c = [], 1
for aid in account_ids:
    cid = acct_to_cust[aid]
    balance = decimal.Decimal(str(round(random.uniform(10000, 200000), 2)))
    for _ in range(random.randint(3, 5)):
        ttype  = random.choice(["CREDIT","DEBIT"])
        amount = money(500, 50000)
        balance = balance + amount if ttype=="CREDIT" else max(decimal.Decimal("0.00"), balance - amount)
        rows.append((
            uid("ATXN",c), aid, cid, ttype, amount, balance,
            maybe(random.choice(DESCS)),
            rand_ts("2024-01-01 00:00:00", "2024-12-31 23:59:59"),
        ))
        c += 1
df_acct_txns = spark.createDataFrame(rows, schema=account_transactions_schema)
print(f"account_txns      : {df_acct_txns.count():>4} rows")

account_txns      :  205 rows


In [11]:
# ── payments ───────────────────────────────────────────────────────────────
rows = []
for i in range(100):
    cid    = random.choice(customer_ids)
    mode   = random.choice(["UPI","UPI","UPI","NEFT","RTGS","IMPS"])
    status = random.choice(["SUCCESS","SUCCESS","SUCCESS","FAILED","PENDING"])
    init   = rand_ts("2024-01-01 00:00:00", "2024-12-31 23:59:59")
    rows.append((
        uid("PAY",i+1), cid, mode,
        money(10, 100000),
        maybe(f"{cid.lower()}@upi") if mode=="UPI" else None,
        maybe(f"merchant{random.randint(1,20)}@upi") if mode=="UPI" else None,
        status,
        random.random() < 0.05,
        init,
        init + timedelta(seconds=random.randint(1,86400)) if status=="SUCCESS" else None,
    ))
df_payments = spark.createDataFrame(rows, schema=payments_schema)
print(f"payments          : {df_payments.count():>4} rows")

payments          :  100 rows


In [12]:
# ── write to data/ ─────────────────────────────────────────────────────────
REPO_ROOT = os.path.dirname(os.path.abspath("generate_bank_data.ipynb"))
DATA      = os.path.join(REPO_ROOT, "data")

(
    df_customers.coalesce(1).write.mode("overwrite")
    .option("header", "true").csv(f"{DATA}/customers")
)
(
    df_repayments.coalesce(1).write.mode("overwrite")
    .option("header", "true").csv(f"{DATA}/loan_repayments")
)
(
    df_card_accounts.coalesce(1).write.mode("overwrite")
    .json(f"{DATA}/card_accounts")
)
(
    df_loan_accounts.coalesce(1).write.mode("overwrite")
    .option("multiLine", "true").json(f"{DATA}/loan_accounts")
)
(
    df_card_txns.write.mode("overwrite")
    .parquet(f"{DATA}/card_transactions")
)
(
    df_bank_accounts.coalesce(1).write.mode("overwrite")
    .orc(f"{DATA}/bank_accounts")
)
(
    df_acct_txns.write.mode("overwrite")
    .orc(f"{DATA}/account_transactions")
)
(
    df_payments.write.mode("overwrite")
    .format("delta").save(f"{DATA}/payments")
)

print("\nAll tables written to data/:")
for name in ["customers","card_accounts","card_transactions","loan_accounts",
             "loan_repayments","bank_accounts","account_transactions","payments"]:
    print(f"  data/{name}")

26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers


26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/08 10:54:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/04/08 10:54:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/08 10:54:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/08 10:54:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/08 10:54:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/08 10:54:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/08 10:54:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers



All tables written to data/:
  data/customers
  data/card_accounts
  data/card_transactions
  data/loan_accounts
  data/loan_repayments
  data/bank_accounts
  data/account_transactions
  data/payments


In [ ]:
# ── validation reads ───────────────────────────────────────────────────────
checks = [
    ("customers (CSV)",
     spark.read.option("header","true").schema(customers_schema).csv(f"{DATA}/customers")),
    ("card_accounts (JSON)",
     spark.read.schema(card_accounts_schema).json(f"{DATA}/card_accounts")),
    ("card_transactions (Parquet)",
     spark.read.schema(card_transactions_schema).parquet(f"{DATA}/card_transactions")),
    ("loan_accounts (JSON multiline)",
     spark.read.option("multiLine","true").schema(loan_accounts_schema).json(f"{DATA}/loan_accounts")),
    ("loan_repayments (CSV)",
     spark.read.option("header","true").schema(loan_repayments_schema).csv(f"{DATA}/loan_repayments")),
    ("bank_accounts (ORC)",
     spark.read.schema(bank_accounts_schema).orc(f"{DATA}/bank_accounts")),
    ("account_transactions (ORC)",
     spark.read.schema(account_transactions_schema).orc(f"{DATA}/account_transactions")),
    ("payments (Delta)",
     spark.read.format("delta").load(f"{DATA}/payments")),
]

print("Validation:")
for label, df in checks:
    print(f"  {label:35s}  {df.count():>5} rows")

In [15]:
spark.stop()